In [7]:
from openai import OpenAI
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import AppendableIndex
import json

openai_client = OpenAI()

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

index = AppendableIndex(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)

def search(query):
    results = index.search(
        query=query,
        num_results=5
    )
    return results

search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the documentation database for relevant results based on a query string.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query to look up in the index"
            }
        },
        "required": [
            "query"
        ]
    }
}

instructions = """
You're a documentation assistant. 

Answer the user question using the documentation knowledge base

Important: When you return results please make at least 3 searches to make sure you explore the topic thoroughly.

Use only facts from the knowledge base when answering.
IMPORTANT: f you cannot find the answer, inform the user.
"""

In [3]:
question = "How do I create a dahsbord in Evidently?"
messages_history = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]

In [4]:
response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages_history,
    tools=[search_tool],
)


In [5]:
response.output

[ResponseFunctionToolCall(arguments='{"query":"create dashboard in Evidently"}', call_id='call_kfAKuIypybeIQf8pLk2oEVBb', name='search', type='function_call', id='fc_0df05743498f93b50069f607645f08819ea1d127100dc54fdb', namespace=None, status='completed')]

In [6]:
def make_call(tool_call):
    arguments = json.loads(tool_call.arguments)
    name = tool_call.name

    if name == 'search':
        result = search(**arguments)
    else: 
        result = 'not found tool "{name}"'
    
    return {
        "type": "function_call_output",
        "call_id": tool_call.call_id,
        "output": json.dumps(result),
    }

In [7]:
messages_history.extend(response.output)

In [9]:
messages_history

[{'role': 'system',
  'content': "\nYou're a documentation assistant. \n\nAnswer the user question using the documentation knowledge base\n\nUse only facts from the knowledge base when answering.\nIMPORTANT: f you cannot find the answer, inform the user.\n"},
 {'role': 'user', 'content': 'How do I create a dahsbord in Evidently?'},
 ResponseFunctionToolCall(arguments='{"query":"create dashboard in Evidently"}', call_id='call_kfAKuIypybeIQf8pLk2oEVBb', name='search', type='function_call', id='fc_0df05743498f93b50069f607645f08819ea1d127100dc54fdb', namespace=None, status='completed')]

In [10]:
for message in response.output:
    if message.type == 'function_call':
        print(f'executing {message.name}({message.arguments})...')
        tool_call_output = make_call(message)
        messages_history.append(tool_call_output)

executing search({"query":"create dashboard in Evidently"})...


In [11]:
len(messages_history)

4

In [12]:
response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages_history,
    tools=[search_tool],
)

response.usage.input_tokens
print(response.output_text)


To create a dashboard in Evidently, follow these steps:

### 1. Create a Project
Before creating a dashboard, ensure that you have connected to Evidently Cloud and created a project.

### 2. Add Tabs
You can organize your panels by adding tabs.

- **To add a Tab**:
   - Enter "Edit" mode on the dashboard (top right corner).
   - Click on the plus sign to add a tab.
   - To create a custom tab, select "empty" and enter a name.

### 3. Add Panels
Panels are where you visualize your data. You can add multiple types of panels such as text panels, counters, pie charts, line plots, etc.

- **To add a Panel**:
   - Enter "Edit" mode on the dashboard.
   - Click on the "Add Panel" button.
   - Follow the prompts to configure the panel.
   - Use the preview to review your setup.
   - Click "Save" and select the tab where you want to add the panel.

### Available Panel Types
- **Text Panels**: For displaying titles or notes.
- **Counter Panels**: To display specific values.
- **Plots (Line, Bar)

Multi-iteration Behavior - we adjust our initial instructions

In [8]:
instructions = """
You're a documentation assistant. 

Answer the user question using the documentation knowledge base

Make 3 iterations:

1) in the first iteration, perform one search
2) in the second interation, analyze the results from the previous search
   and perform 2 more searches
3) synthesise the results into the output

IMPORTANT: at each step, give an explanation of why you want to perform 
search for this particular search query. It should be 2-3 sentences explaining
the logic of your decision.

Use only facts from the knowledge base when answering.
If you cannot find the answer, inform the user.

Our knowledge base is entirely about Evidently, so you don't need to 
include the word 'evidently' in search results
"""

In [14]:
message_history = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]

iteration_number = 1

while True:
    response = openai_client.responses.create(
        model='gpt-4o-mini',
        input=message_history,
        tools=[search_tool],
    )

    print(f'iteraration number {iteration_number}...') 
    message_history.extend(response.output)

    has_function_calls = False

    for message in response.output:
        if message.type == 'function_call':
            print(f'executing {message.name}({message.arguments})...')
            tool_call_output = make_call(message)
            message_history.append(tool_call_output)
            has_function_calls = True

        if message.type == 'message':
            text = message.content[0].text
            print('ASSISTANT:', text)

    iteration_number = iteration_number + 1
    print()
    
    if not has_function_calls:
        break

iteraration number 1...
ASSISTANT: To provide a comprehensive answer, I'll start by searching for information on how to create a dashboard. This will help ensure I'm referencing the specific steps and considerations for dashboard creation in the platform.

Let's perform the first search for "create a dashboard."
executing search({"query":"create a dashboard"})...

iteraration number 2...
ASSISTANT: The first search on "create a dashboard" yielded useful information about dashboard creation, including how to add panels and tabs. The dashboard allows the visualization of project data, and different types of panels can be included. This establishes the foundation for further understanding.

Next, to provide more detailed steps and example codes, I will perform two additional searches: one for "dashboard management" and another for "adding tabs and panels." This will clarify different dashboard management aspects and provide varied methods for implementation.

Let’s proceed with the next s

In [15]:
def add_entry(filename, title, description, content):
    entry = {
        'start': 0,
        'content': content,
        'title': title,
        'description': description,
        'filename': filename,
    }
    index.append(entry)
    return "OK"

add_entry_tool = {
    "type": "function",
    "name": "add_entry",
    "description": "Add a new documentation entry to the index.",
    "parameters": {
        "type": "object",
        "properties": {
            "filename": {
                "type": "string",
                "description": "The source filename associated with the entry"
            },
            "title": {
                "type": "string",
                "description": "The title of the documentation entry"
            },
            "description": {
                "type": "string",
                "description": "A short description summarizing the entry"
            },
            "content": {
                "type": "string",
                "description": "The full content of the documentation entry"
            }
        },
        "required": [
            "filename",
            "title",
            "description",
            "content"
        ]
    }
}

In [17]:
# Updated make_call

def make_call(tool_call):
    arguments = json.loads(tool_call.arguments)
    name = tool_call.name

    if name == 'search':
        result = search(**arguments)
    elif name == 'add_entry':
        result = add_entry(**arguments)
    else: 
        result = 'not found tool "{name}"'
    
    return {
        "type": "function_call_output",
        "call_id": tool_call.call_id,
        "output": json.dumps(result),
    }

In [18]:
message_history = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]
tools = [search_tool, add_entry_tool]

iteration_number = 1

while True:
    response = openai_client.responses.create(
        model='gpt-4o-mini',
        input=message_history,
        tools=tools,
    )

    print(f'iteraration number {iteration_number}...') 
    message_history.extend(response.output)

    has_function_calls = False

    for message in response.output:
        if message.type == 'function_call':
            print(f'executing {message.name}({message.arguments})...')
            tool_call_output = make_call(message)
            message_history.append(tool_call_output)
            has_function_calls = True

        if message.type == 'message':
            text = message.content[0].text
            print('ASSISTANT:', text)

    iteration_number = iteration_number + 1
    print()
    
    if not has_function_calls:
        break

iteraration number 1...
ASSISTANT: To assist you effectively in creating a dashboard, I will first search for documentation on dashboard creation. This will help me gather foundational information regarding the steps, best practices, and any prerequisites needed for building a dashboard.

Let's start by looking up "create dashboard" in the documentation.
executing search({"query":"create dashboard"})...

iteraration number 2...
ASSISTANT: The initial search results provide detailed instructions on how to create a dashboard by adding panels and organizing them with tabs. It outlines the steps to enter edit mode, create tabs, add various panel types (including text, counters, and plots), and the importance of having reports with evaluation results for panel population.

To gain a more comprehensive understanding, I will conduct additional searches focused on specific aspects of dashboard creation, such as panel customization options and the available metrics for dashboards. This will hel

In [19]:
message_history.append(
    {"role": "user", "content": "add this content to our database"}
)

In [20]:
while True:
    response = openai_client.responses.create(
        model='gpt-4o-mini',
        input=message_history,
        tools=tools,
    )

    print(f'iteraration number {iteration_number}...') 
    message_history.extend(response.output)

    has_function_calls = False

    for message in response.output:
        if message.type == 'function_call':
            print(f'executing {message.name}({message.arguments})...')
            tool_call_output = make_call(message)
            message_history.append(tool_call_output)
            has_function_calls = True

        if message.type == 'message':
            text = message.content[0].text
            print('ASSISTANT:', text)

    iteration_number = iteration_number + 1
    print()
    
    if not has_function_calls:
        break


iteraration number 4...
executing add_entry({"filename":"docs/platform/dashboard_creation_guide.mdx","title":"How to Create a Dashboard","description":"Comprehensive guide on creating and managing dashboards with panels","content":"### How to Create a Dashboard\n\n1. **Initial Setup**:\n   - Ensure you are connected to the platform and have created a project to house your dashboards.\n\n2. **Adding Tabs**:\n   - Enter **Edit mode** on the dashboard.\n   - Click the \" + \" sign to add a new tab or select \"empty\" for a custom tab name.\n   - Consider using pre-built tabs to simplify the initial setup.\n\n3. **Adding Panels**:\n   - Click the **Add Panel** button to create panels which can display various visualization types, including:\n     - **Text panels**\n     - **Counters**\n     - **Pie charts**\n     - **Line plots**\n     - **Bar plots (grouped/staked)**\n\n4. **Configuring Panels**:\n   - While adding a panel:\n     - Select metrics matching the names from the reports logged

In [21]:
index.docs[-1]

{'start': 0,
 'content': '### How to Create a Dashboard\n\n1. **Initial Setup**:\n   - Ensure you are connected to the platform and have created a project to house your dashboards.\n\n2. **Adding Tabs**:\n   - Enter **Edit mode** on the dashboard.\n   - Click the " + " sign to add a new tab or select "empty" for a custom tab name.\n   - Consider using pre-built tabs to simplify the initial setup.\n\n3. **Adding Panels**:\n   - Click the **Add Panel** button to create panels which can display various visualization types, including:\n     - **Text panels**\n     - **Counters**\n     - **Pie charts**\n     - **Line plots**\n     - **Bar plots (grouped/staked)**\n\n4. **Configuring Panels**:\n   - While adding a panel:\n     - Select metrics matching the names from the reports logged in your project.\n     - Use the filtering options to specify the data accurately.\n     - Adjust display parameters like title, size (full or half), and choose the type of visualization.\n   - Save the panel 

In [ ]:
# Q & A Loop Allows Interaction
tools = [search_tool, add_entry_tool]

message_history = [
    {"role": "system", "content": instructions},
]

iteration_number = 1

# Q&A loop
while True:
    user_prompt = input('You:')
    if user_prompt.lower().strip() == 'stop':
        break

    message_history.append({"role": "user", "content": user_prompt})

    # tool-call loop
    while True:
        response = openai_client.responses.create(
            model='gpt-4o-mini',
            input=message_history,
            tools=tools,
        )
    
        print(f'iteraration number {iteration_number}...') 
        message_history.extend(response.output)
    
        has_function_calls = False
    
        for message in response.output:
            if message.type == 'function_call':
                print(f'executing {message.name}({message.arguments})...')
                tool_call_output = make_call(message)
                message_history.append(tool_call_output)
                has_function_calls = True
    
            if message.type == 'message':
                text = message.content[0].text
                print('ASSISTANT:', text)
    
        iteration_number = iteration_number + 1
        print()
        
        if not has_function_calls:
            break


iteraration number 1...
ASSISTANT: It looks like you didn't include a question. Please provide a topic or a specific question you'd like me to assist you with!



In [3]:
class Agent:

    def __init__(self, llm_client, model, instructions, tools):
        self.llm_client = llm_client
        self.model = model
        self.instructions = instructions
        self.tools = tools

    def make_call(self, tool_call):
        arguments = json.loads(tool_call.arguments)
        name = tool_call.name
    
        if name == 'search':
            result = search(**arguments)
        elif name == 'add_entry':
            result = add_entry(**arguments)
        else:
            result = 'not found tool "{name}"'
        
        return {
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": json.dumps(result),
        }   

    def loop(self, user_prompt, message_history=None):
        if not message_history:
            message_history = [
                {"role": "system", "content": self.instructions},
            ]
            
        message_history.append({"role": "user", "content": user_prompt})

        iteration_number = 0
    
        while True:
            response = self.llm_client.responses.create(
                model=self.model,
                input=message_history,
                tools=self.tools,
            )
        
            print(f'iteraration number {iteration_number}...') 
            message_history.extend(response.output)
        
            has_function_calls = False
        
            for message in response.output:
                if message.type == 'function_call':
                    print(f'executing {message.name}({message.arguments})...')
                    tool_call_output = self.make_call(message)
                    message_history.append(tool_call_output)
                    has_function_calls = True
        
                if message.type == 'message':
                    text = message.content[0].text
                    print('ASSISTANT:', text)
        
            iteration_number = iteration_number + 1
            print()
            
            if not has_function_calls:
                break

        return message_history        

    def qna(self):
        message_history = [
            {"role": "system", "content": instructions},
        ]
        
        iteration_number = 1

        # Q&A loop
        while True:
            user_prompt = input('You:')
            if user_prompt.lower().strip() == 'stop':
                break
            
            message_history = self.loop(user_prompt, message_history)


In [9]:
tools = [search_tool, add_entry_tool]

message_history = [
    {"role": "system", "content": instructions},
]


agent = Agent(
    llm_client=OpenAI(),
    model='gpt-4o-mini',
    instructions=instructions,
    tools=tools
)

NameError: name 'add_entry_tool' is not defined

#### Think about structured response approach
Some LLMs don't natively support structured output. For these models, using a fake tool call is still the only way to get structured responses. Even for OpenAI, this approach lets the LLM decide when it's ready to produce structured output, rather than forcing it on every iteration.